# Colab training preflight

Run this diagnostic after connecting to the intended runtime. It installs nothing, reads no secrets, downloads no model and starts no training. It reports only selected hardware/package facts. A GPU selection in the UI must still be verified here. Copy this notebook before adding a task-specific, version-pinned training recipe.

Continue with data validation, a bounded training smoke run, evaluation, checkpoint restoration and durable export. See the skill references for the operating procedure.

In [ ]:
import importlib.metadata
import json
import platform
import shutil
import subprocess

report = {"python": platform.python_version(), "packages": {}}
for package in ("torch", "transformers", "trl", "peft", "accelerate", "bitsandbytes", "datasets"):
    try:
        report["packages"][package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        report["packages"][package] = None
scratch = "/content" if __import__("pathlib").Path("/content").is_dir() else "."
report["scratch_free_gib"] = round(shutil.disk_usage(scratch).free / 2**30, 2)
if shutil.which("nvidia-smi"):
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=10, check=False,
        )
        report["nvidia_smi"] = result.stdout.strip() if result.returncode == 0 else "command_failed"
    except (OSError, subprocess.TimeoutExpired):
        report["nvidia_smi"] = "unavailable_or_timeout"
else:
    report["nvidia_smi"] = "not_installed"
try:
    import torch
    report["torch_cuda_build"] = torch.version.cuda
    report["cuda_available"] = torch.cuda.is_available()
    if report["cuda_available"]:
        report["device"] = torch.cuda.get_device_name(0)
        report["bf16_supported"] = bool(torch.cuda.is_bf16_supported())
except ImportError:
    report["cuda_available"] = None
    report["torch_status"] = "not_importable"
print(json.dumps(report, indent=2))


## Interpret the result

`cuda_available: false` means GPU training is not ready, even if a GPU name appears in the UI. `null` means PyTorch could not be imported. Install the tested recipe's compatible environment deliberately, restart if required, and rerun this cell. BF16 support is a capability check, not a guarantee that every model/kernel supports it. Scratch is ephemeral; verify a durable output destination before starting the training cells you add. These diagnostics alone do not validate model quality or prove that a planned model fits.